# 00 — Setup and data check

Run this first in a new Colab session. It connects to the response sheet, loads the cluster configuration
(`src/config/clusters.json`, including the committee parameters), and shows what was loaded.

**You need:** Viewer access to the response sheet, and its Spreadsheet ID (the long string in the sheet URL between `/d/` and `/edit`).

In [ ]:
%pip install -q gspread pandas matplotlib

In [ ]:
SPREADSHEET_ID = 'PASTE_YOUR_SPREADSHEET_ID_HERE'   # <- the long string in the sheet URL
REPO_REF = 'main'                                   # git branch/tag for the helper code and cluster config

import os, urllib.request
if not os.path.exists('hpc_analysis.py'):            # in Colab: fetch the helper modules once per session
    for f in ['sheets_client.py', 'hpc_analysis.py', 'hpc_plots.py', 'code_capabilities.json']:
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/svaradh/hpc-questionnaire/{REPO_REF}/analysis/{f}', f)

import pandas as pd
from sheets_client import load_data, load_config
import hpc_analysis as ha
import hpc_plots as hp
pd.set_option('display.max_colwidth', 120)

dfs = load_data(SPREADSHEET_ID)
cfg = load_config(ref=REPO_REF)
wl = ha.workloads(dfs, cfg)
cov = ha.coverage(dfs)
print(f"{cov['groups']} groups responded; {cov['workloads']} workloads described "
      f"({cov['groups_with_workloads']} groups with at least one).")

## Tabs loaded

In [ ]:
from sheets_client import summarise_sheets
summarise_sheets(dfs)

## Cluster configuration and committee parameters

Capacity figures in the other notebooks are computed from these. `None` means *not yet set by the committee*;
analyses that need such a value say so instead of guessing.

In [ ]:
rows = [{'cluster': c['name'], 'CPU cores': c['cpuCores'], 'GPUs': c['gpuCount'],
         'home TB': c['storage']['homeTB'], 'scratch TB': c['storage']['scratchTB'],
         'scratch fraction': c['storage']['scratchFraction']} for c in cfg['clusters']]
display(pd.DataFrame(rows))
pd.Series(cfg['committeeParameters']).drop('_comment', errors='ignore')

## Submissions

In [ ]:
dfs['Submissions'][['submission_id', 'pi_name', 'group_name', 'department', 'submitted_at', 'questionnaire_version']]